# County Rainfall and Planting Advisory

**PLP Data Analytics Assignment 2**

This project analyzes monthly rainfall data for five Kenyan counties:
Nairobi, Nakuru, Kisumu, Mombasa, and Machakos.

The analysis uses Python and NumPy to calculate rainfall statistics,
identify dry months, and recommend two-month planting windows.

In [1]:
import numpy as np
import csv
import sys

print("Python:", sys.version.split()[0])
print("NumPy version:", np.__version__)
print("Environment ready!")

Python: 3.15.2
NumPy version: 2.5.5
Environment ready!


## 1. Load the Rainfall Dataset

The dataset contains twelve months of rainfall readings
for five counties. Machakos contains two invalid readings.

In [2]:
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

months = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70]
]

machakos_raw = [
    "48", "35", "88", "N/A", "140", " 20 ",
    "9", "12", "", "45", "182", "96"
]

print("Counties:", counties)
print("Number of months:", len(months))
print("Machakos raw readings:", machakos_raw)

Counties: ['Nairobi', 'Nakuru', 'Kisumu', 'Mombasa', 'Machakos']
Number of months: 12
Machakos raw readings: ['48', '35', '88', 'N/A', '140', ' 20 ', '9', '12', '', '45', '182', '96']


## 2. Clean Machakos Rainfall Data

The cleaning function converts valid readings to integers.
It catches invalid values using `except ValueError` and
replaces them with the rounded mean of valid readings.

In [3]:
def clean_machakos(data):
    cleaned_values = []
    valid_values = []
    imputed_months = []

    for i, value in enumerate(data):
        try:
            rainfall_value = int(value.strip())
            cleaned_values.append(rainfall_value)
            valid_values.append(rainfall_value)

        except ValueError:
            cleaned_values.append(None)
            imputed_months.append(months[i])

    average = round(np.mean(valid_values))

    for i, value in enumerate(cleaned_values):
        if value is None:
            cleaned_values[i] = average

    print("Average rainfall used:", average, "mm")
    print("Imputed months:", imputed_months)
    print("Cleaned Machakos readings:", cleaned_values)

    return cleaned_values


machakos_clean = clean_machakos(machakos_raw)

Average rainfall used: 68 mm
Imputed months: ['April', 'September']
Cleaned Machakos readings: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


## 3. Construct the Combined NumPy Array

Combine the four complete county rows with the cleaned
Machakos row to create a 5 × 12 NumPy array.

In [4]:
all_rows = clean_rows + [machakos_clean]
rainfall = np.array(all_rows)

print("Combined Rainfall Dataset:")
print(rainfall)

print("\nArray shape:", rainfall.shape)
print("Array data type:", rainfall.dtype)

Combined Rainfall Dataset:
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]

Array shape: (5, 12)
Array data type: int64


## 4. Annual Rainfall Totals and Monthly Averages

NumPy calculates annual rainfall totals for each county
and monthly average rainfall across the five counties.

In [5]:
annual_totals = np.sum(rainfall, axis=1)
monthly_averages = np.mean(rainfall, axis=0)

print("ANNUAL RAINFALL TOTALS")

for county, total in zip(counties, annual_totals):
    print(f"{county}: {total} mm")

print("\nMONTHLY RAINFALL AVERAGES")

for month, average in zip(months, monthly_averages):
    print(f"{month}: {average:.2f} mm")

ANNUAL RAINFALL TOTALS
Nairobi: 1045 mm
Nakuru: 916 mm
Kisumu: 1371 mm
Mombasa: 1075 mm
Machakos: 811 mm

MONTHLY RAINFALL AVERAGES
January: 47.00 mm
February: 47.60 mm
March: 91.00 mm
April: 164.40 mm
May: 171.20 mm
June: 67.60 mm
July: 56.60 mm
August: 58.20 mm
September: 64.20 mm
October: 67.80 mm
November: 127.40 mm
December: 80.60 mm


## 5. Identify the Wettest Month per County

Use `np.argmax` to identify the month with the highest
rainfall reading in each county.

In [6]:
wettest_indices = np.argmax(rainfall, axis=1)

wettest_months = np.array(months)[wettest_indices]

wettest_rainfall = np.max(rainfall, axis=1)

print("WETTEST MONTH PER COUNTY")

for county, month, value in zip(
    counties, wettest_months, wettest_rainfall
):
    print(f"{county}: {month} ({value} mm)")

WETTEST MONTH PER COUNTY
Nairobi: April (219 mm)
Nakuru: April (145 mm)
Kisumu: April (240 mm)
Mombasa: May (240 mm)
Machakos: November (182 mm)


## 6. Count Dry Months per County

A dry month is defined as a month with rainfall below 50 mm.
A NumPy Boolean mask identifies these months.

In [7]:
dry_mask = rainfall < 50

dry_month_counts = np.sum(dry_mask, axis=1)

print("DRY MONTHS PER COUNTY")

for county, count in zip(counties, dry_month_counts):
    print(f"{county}: {count} dry months")

DRY MONTHS PER COUNTY
Nairobi: 4 dry months
Nakuru: 3 dry months
Kisumu: 0 dry months
Mombasa: 2 dry months
Machakos: 6 dry months


## 7. Calculate the Best Two-Month Planting Window

Calculate combined rainfall for every pair of consecutive
months. Identify the pair with the highest combined rainfall
for each county using NumPy.

In [8]:
two_month_totals = rainfall[:, :-1] + rainfall[:, 1:]

best_window_indices = np.argmax(two_month_totals, axis=1)

best_window_totals = np.max(two_month_totals, axis=1)

planting_windows = [
    f"{months[i]} - {months[i + 1]}"
    for i in best_window_indices
]

print("BEST TWO-MONTH PLANTING WINDOWS")

for county, window, total in zip(
    counties, planting_windows, best_window_totals
):
    print(f"{county}: {window} ({total} mm)")

BEST TWO-MONTH PLANTING WINDOWS
Nairobi: April - May (395 mm)
Nakuru: April - May (265 mm)
Kisumu: April - May (420 mm)
Mombasa: April - May (390 mm)
Machakos: November - December (278 mm)


## 8. Export the Rainfall Summary CSV

Save the required county statistics into
`rainfall_summary.csv` using Python's CSV module.

In [9]:
with open(
    "rainfall_summary.csv",
    "w",
    newline="",
    encoding="utf-8"
) as file:

    writer = csv.writer(file)

    writer.writerow([
        "county",
        "annual_total_mm",
        "wettest_month",
        "dry_months",
        "planting_window"
    ])

    writer.writerows(zip(
        counties,
        annual_totals,
        wettest_months,
        dry_month_counts,
        planting_windows
    ))

print("rainfall_summary.csv created successfully!")

rainfall_summary.csv created successfully!


## Planting Advisory

The rainfall analysis indicates that April–May is the strongest
two-month rainfall window for Nairobi, Nakuru, Kisumu, and Mombasa.
Machakos has its strongest consecutive rainfall window in
November–December. Farmers may consider these periods when planning
planting activities, while also considering crop type, soil moisture,
land preparation, and local weather forecasts. Kisumu recorded the
highest annual rainfall at 1,371 mm, while Nakuru recorded the lowest
at 916 mm. These findings provide a general planning guide rather
than a guarantee of suitable growing conditions.

## Data Quality Notes

Machakos had two missing or invalid rainfall entries: April (`N/A`)
and September (blank). These values were replaced with the mean of
the ten valid monthly readings, approximately 68 mm. This imputation
allows the county data to be analyzed consistently, but introduces
some uncertainty into Machakos's annual total, monthly averages,
and planting recommendations. The analysis is based on one year's
monthly rainfall data and should be supplemented with current
weather forecasts and local agricultural guidance.